# 01. 학습 데이터 생성과 LoRA 학습 (논문 3.3절, 표 1)

| 단계 | 입력 | 생성되는 데이터 |
|---|---|---|
| 학습 데이터 생성 | 칭찬·격려 템플릿 | 완료/미완료 × 인증 회차를 조합한 **500쌍**(칭찬 250, 격려 250) instruction 데이터 |
| LoRA 학습 | 500쌍 | Gemma-3n-E2B-it에 LoRA를 적용한 메시지 생성 모델 |
| 생성 확인 | 관심 가수 영상 제목 + 인증 정보 | LoRA 학습 이후 동기부여 메시지 |

실행 환경: Google Colab, NVIDIA A100(40GB). 모델은 Google Drive의 `gemma-3n-E2b-it` 폴더에서 로드합니다.

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes #라이브러리 설치

In [ ]:
!pip install -q datasets transformers peft accelerate bitsandbytes scikit-learn

## 1. 학습 데이터 생성

템플릿에서 무작위로 응답을 골라 `숙제를 완료한 / 완료하지 못한 아이에게 메세지 생성: {인증 정보}` → `{응원 문장}` 형식의 쌍을 만듭니다.
템플릿 원문은 `data/training_templates.json`에도 정리되어 있습니다.

In [ ]:
import random
from datasets import Dataset

# ✅ 칭찬 메시지 템플릿
praise_templates = [
    "BoA의 Crazier처럼 오늘 너의 집중력도 미쳤다! 숙제 완벽 클리어 🎯",
    "Beautiful Life 같은 하루였네! 숙제까지 끝냈다니 완벽 그 자체야 🌟",
    "와! 아이유 'Celebrity'처럼 너만의 매력으로 숙제를 완벽하게 해냈네! 정말 자랑스러워.",
    "에스파 'Whiplash'처럼, 숙제를 휙휙 날려버리는 너의 집중력에 감탄했어! 최고야!",
    "IVE 'I AM'처럼, 너 자신을 믿고 숙제를 해낸 모습이 정말 멋있어! 넌 할 수 있다는 걸 증명했네.",
    "아이브 'REBEL HEART'처럼, 숙제에 대한 열정으로 가득 찬 너를 응원해! 넌 최고야.",
    "아이유 'Love wins all'처럼, 너의 노력은 결국 빛을 발할 거야! 숙제 완료를 축하해.",
    "전소미 'CLOSER'처럼, 숙제에 더욱 가까이 다가가면서 성장하는 너를 응원해! 넌 정말 대단해.",
    "KEY 'HUNTER'처럼, 숙제를 목표 삼아 집중하는 너의 모습이 인상적이었어! 넌 최고야.",
    "Woochi처럼, 숙제를 정복하고 새로운 레벨업을 달성했네! 축하해!",
    "박다혜 & MAKTUB '시작의 아이'처럼, 완벽한 시작과 마무리를 보여주는 너를 응원해!",
    "원피스 루피처럼, 꿈을 향해 나아가는 너의 열정을 숙제에서도 보여줬네! 정말 멋있어.",
    "아이유 콘서트 플레이리스트처럼, 숙제를 너만의 무대로 만들어버린 너를 응원해! 즐거웠지?",
    "에스파 노래처럼, 숙제를 통해 너만의 강렬한 에너지를 발휘했네! 넌 최고야.",
    "IVE 노래처럼, 너의 'ATTITUDE'로 숙제를 완벽하게 해냈어! 정말 멋있어.",
    "아이유 'Blueming'처럼, 너의 숙제는 아름답게 피어났어! 정말 잘했어.",
    "아이브 'Be Alright'처럼, 숙제를 완벽하게 해내서 기분이 정말 좋을 거야! 축하해.",
    "아이브 'After LIKE'처럼, 숙제를 완벽하게 해내서 너를 더욱 응원하게 돼! 넌 최고야.",
    "아이유의 킬링보이스처럼, 숙제도 완벽하게 소화해내는 너를 응원해!",
    "에스파 플레이리스트처럼, 다양한 숙제를 완벽하게 해내는 너를 응원해!",
    "아이유 플레이리스트처럼, 너에게 힘이 되는 숙제를 완벽하게 해냈네! 넌 최고야.",
    "Woochi the Wayfarer처럼, 숙제를 통해 새로운 가능성을 발견했네! 축하해.",
    "박다혜 노래처럼, 완벽한 숙제 완료를 축하해! 넌 할 수 있어!",
    "전소미의 노래처럼, 숙제에 더욱 가까이 다가가면서 성장하는 너를 응원해! 넌 정말 대단해.",
    "KEY의 노래처럼, 숙제를 목표 삼아 집중하는 너의 모습이 인상적이었어! 넌 최고야.",
    "아이유의 'Love wins all'처럼, 너의 노력은 결국 빛을 발할 거야! 숙제 완료를 축하해.",
    "IVE의 'I AM'처럼, 너 자신을 믿고 숙제를 해낸 모습이 정말 멋있어! 넌 할 수 있다는 걸 증명했네.",
    "에스파의 'Dirty Work'처럼, 숙제를 깔끔하게 처리하는 너의 능력이 대단해!",
    "아이브의 'I WANT'처럼, 숙제를 완벽하게 해내고 싶은 너의 열정이 느껴져!",
    "아이유의 'Celebrity'처럼, 너만의 개성으로 숙제를 완벽하게 해냈네! 정말 자랑스러워.",
    "Woochi처럼, 숙제를 정복하고 새로운 레벨업을 달성했네! 축하해!",
    "박다혜 & MAKTUB '시작의 아이'처럼, 완벽한 숙제 완료를 축하해! 넌 최고야.",
]


# ✅ 격려 메시지 템플릿
encouragement_templates = [
    "Beautiful Life처럼 오늘도 의미 있는 하루였어. 숙제는 내일 천천히 해도 괜찮아 🌿",
    "악동 김블루 영상처럼 즐겁게 배우는 게 제일 중요해. 오늘 못한 건 내일 보충하자 😄",
    "랍온어비트 마술사처럼 네 안에 숨겨진 재능은 아직 많아. 숙제는 서두르지 않아도 돼 ✨",
    "침착맨 라디오처럼 긴장되는 순간도 있지만, 그게 성장의 일부야 📚",
    "목청으로 챌린저들 매장 시키듯, 내일은 숙제도 시원하게 끝내자 🔥",
    "BoA의 Crazier처럼 내일은 더 에너지 넘치게 달려보자 💫",
    "아이유의 'Love wins all'처럼, 아직 숙제가 남았지만, 결국엔 네 노력도 빛을 발할 거야! 힘내!",
    "에스파 'Whiplash'처럼, 숙제에 대한 부담감을 휙휙 날려버리고, 다시 시작해보자! 넌 할 수 있어.",
    "IVE 'I AM'처럼, 너 자신을 믿고 다시 숙제를 해나가길 응원해! 넌 이미 충분히 멋져.",
    "아이브 'REBEL HEART'처럼, 숙제에 대한 반항심 대신, 너만의 방식으로 해나가길 응원해!",
    "아이유 콘서트 플레이리스트처럼, 숙제를 너만의 무대로 만들어봐! 즐겁게 해나가길 응원해.",
    "전소미 'CLOSER'처럼, 숙제에 조금씩 더 다가가면서 성장하는 너를 응원해! 넌 할 수 있어!",
    "KEY 'HUNTER'처럼, 남은 숙제를 목표 삼아 집중해봐! 너의 능력을 믿어.",
    "Woochi처럼, 잠시 길을 잃어도 괜찮아. 다시 시작하면 돼! 스스로 선택하는 너를 응원해.",
    "박다혜 & MAKTUB '시작의 아이'처럼, 지금부터라도 다시 시작하면 돼! 늦지 않았어.",
    "원피스 루피처럼, 포기하지 않고 숙제에 도전하는 너의 열정을 응원해! 끈기를 보여줘.",
    "아이유 'Blueming'처럼, 아직 숙제가 피어나지 않았지만, 곧 아름다운 결과가 있을 거야!",
    "아이브 'Be Alright'처럼, 숙제가 힘들겠지만, 괜찮아질 거야! 힘내!",
    "아이브 'After LIKE'처럼, 숙제를 완벽하게 해내고 싶은 너의 열정을 응원해!",
    "아이유의 킬링보이스처럼, 숙제도 완벽하게 소화해낼 수 있을 거야! 힘내!",
    "에스파 플레이리스트처럼, 다양한 숙제를 극복해나가는 너를 응원해!",
    "아이유 플레이리스트처럼, 너에게 힘이 되는 숙제를 찾고 즐겁게 해나가길 바라! 넌 최고야.",
    "Woochi the Wayfarer처럼, 숙제를 통해 새로운 가능성을 발견할 수 있을 거야! 힘내!",
    "박다혜 노래처럼, 시작이 반이라는 마음으로 남은 숙제를 마무리해봐! 넌 할 수 있어!",
    "전소미의 노래처럼, 숙제에 조금 더 가까이 다가가면서 성장하는 너를 응원해! 넌 할 수 있어!",
    "KEY의 노래처럼, 남은 숙제를 목표 삼아 집중해봐! 너의 능력을 믿어.",
    "아이유의 'Love wins all'처럼, 너의 노력은 결국 빛을 발할 거야! 힘내!",
    "IVE의 'I AM'처럼, 너 자신을 믿고 다시 숙제를 해나가길 응원해! 넌 이미 충분히 멋져.",
    "에스파의 'Dirty Work'처럼, 숙제를 깔끔하게 마무리할 수 있도록 응원할게!",
    "아이브의 'I WANT'처럼, 숙제를 완벽하게 해내고 싶은 너의 열정을 응원해!",
    "아이유의 'Celebrity'처럼, 너만의 개성으로 숙제를 해나가길 응원해!",
    "Woochi처럼, 숙제를 통해 새로운 레벨업을 향해 나아가길 응원해!",
    "박다혜 & MAKTUB '시작의 아이'처럼, 지금부터라도 다시 시작하면 돼! 늦지 않았어.",
    "전소미의 'CLOSER'처럼, 숙제에 조금씩 더 다가가면서 성장하는 너를 응원해! 넌 할 수 있어!",
    "KEY의 'HUNTER'처럼, 남은 숙제를 목표 삼아 집중해봐! 너의 능력을 믿어.",
    "아이유의 'Love wins all'처럼, 결국엔 네 노력도 빛을 발할 거야! 힘내!",
]

# ✅ 250쌍씩 praise & encouragement 생성
praise_data = [
    {
        "type": "attendance",
        "content": f"이번 챌린지 {i%30+1}회차 인증 (누적 {i%30}회 인증)",
        "complete": "1",
        "response": random.choice(praise_templates)
    }
    for i in range(250)
]

encouragement_data = [
    {
        "type": "attendance",
        "content": f"이번 챌린지 {i%30+1}회차 인증 (누적 {i%30}회 인증)",
        "complete": "0",
        "response": random.choice(encouragement_templates)
    }
    for i in range(250)
]

# ✅ 데이터 합치고 섞기
combined_data = praise_data + encouragement_data
random.shuffle(combined_data)

# ✅ 프롬프트 + 응답 형식으로 변환
instruction_dataset = [
    {
        "input": f"{'숙제를 완료한' if d['complete'] == '1' else '숙제를 완료하지 못한'} 아이에게 메세지 생성: {d['content']}",
        "output": d["response"]
    }
    for d in combined_data
]

# ✅ Dataset 객체 생성
dataset = Dataset.from_list(instruction_dataset)

# 결과 확인 (선택적 출력)
print(f"총 샘플 수: {len(dataset)}")
print("예시:\n", dataset[0])
# 🤖 Dataset 생성
dataset = Dataset.from_list(instruction_dataset)

print(dataset.column_names)

총 샘플 수: 500
예시:
 {'input': '숙제를 완료하지 못한 아이에게 메세지 생성: 이번 챌린지 23회차 인증 (누적 22회 인증)', 'output': '악동 김블루 영상처럼 즐겁게 배우는 게 제일 중요해. 오늘 못한 건 내일 보충하자 😄'}
['input', 'output']


## 2. LoRA 설정과 학습

`r=16`, `lora_alpha=32`, `lora_dropout=0.5`, `target_modules=["q_proj", "v_proj"]`, epoch 4, batch size 1, gradient accumulation 4.
학습 문장은 `<start_of_turn>user` / `<start_of_turn>model` 구분자로 구성하고, 최대 길이 128로 토큰화합니다.

In [ ]:
#gemma 3B 짜리
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from pathlib import Path
import torch
# 로컬 디렉토리 경로
model_path = Path("/content/drive/MyDrive/gemma-3n-E2b-it")  # 문자열 또는 Path 객체

device = "cuda" if torch.cuda.is_available() else "cpu"
# Tokenizer와 Model을 로컬에서 불러옴
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32, # <-- Change to torch.float16
    device_map="auto",
    local_files_only=True
)

# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# PEFT 모델 구성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,374,528 || all params: 5,443,812,800 || trainable%: 0.0804


In [ ]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling, default_data_collator
from transformers import AutoTokenizer, BitsAndBytesConfig
import torch

# 🧠 템플릿 포맷 함수
special_tokens = {
    "additional_special_tokens": ["<start_of_turn>", "<end_of_turn>"]
}
tokenizer.add_special_tokens(special_tokens)              # 토크나이저에 추가
model.resize_token_embeddings(len(tokenizer))             # 모델 임베딩 크기 재조정

# ✅ 템플릿 포맷 함수 (instruction → text 하나로 합침)
def format_instruction(example):
    return {
        "text": f"<start_of_turn>user\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    }

formatted_dataset = dataset.map(format_instruction)

# 🧼 Tokenize 함수
def tokenize(example):
    tokenized = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # 너무 짧으면 안 됨
    )
    input_ids = tokenized["input_ids"]

    # <start_of_turn>model 이후만 label로 사용
    model_token = tokenizer.convert_tokens_to_ids("<start_of_turn>")
    model_index = input_ids.index(model_token) if model_token in input_ids else 0

    labels = [-100] * model_index + input_ids[model_index:]
    tokenized["labels"] = labels
    return tokenized

# 🧪 Tokenize 적용
tokenized_dataset = formatted_dataset.map(tokenize, remove_columns=formatted_dataset.column_names)

# ⚙️ 학습 설정
training_args = TrainingArguments(
    output_dir="./results",                     # ✅ 모델 학습 결과(가중치, 로그 등)를 저장할 디렉토리 경로
    per_device_train_batch_size=1,              # ✅ 각 디바이스(GPU/CPU)당 처리할 배치 사이즈 (작은 모델일수록 작게)
    gradient_accumulation_steps=4,              # ✅ 실제 배치 사이즈를 키우기 위한 누적 단계 수
                                                #    → 실 배치 사이즈 = 1 × 4 = 4
    num_train_epochs=4,                         # ✅ 학습 데이터를 몇 번 반복해서 학습할지 (에폭 수)
    logging_steps=50,                            # ✅ 학습 로그를 출력할 간격 (스텝 단위)
    save_steps=100,                              # ✅ 모델을 저장할 간격 (스텝 단위)
    fp16=True,                                  # ✅ 16-bit float 연산을 사용해 메모리 사용량과 속도를 최적화
                                                #    (단, GPU가 필요하고 지원돼야 함)
    report_to="none"                            # ✅ 로깅 툴 사용 안 함 (예: wandb, tensorboard 사용 안 함)
)

# 📦 모델 준비
model.train()
model.config.use_cache = False
model.print_trainable_parameters()  # ✅ LoRA 파라미터만 학습되는지 확인

# ⚙️ Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
)

# 🚀 학습 시작
trainer.train()

## 3. 학습 이후 메시지 생성 확인

YouTube 검색으로 관심 가수 영상 목록(`final_combined_results`)을 만든 뒤, 그 중 하나를 무작위로 골라 프롬프트에 넣어 메시지를 생성합니다.

In [ ]:
import os
import requests
import json  # 🔹 추가

from google.colab import userdata
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API')  # Colab 보안 비밀에 등록한 키
YOUTUBE_URL = "https://www.googleapis.com/youtube/v3/search"

def get_artist_music_videos(artist_name: str, max_results: int = 50, order_by: str = "relevance"):
    if not YOUTUBE_API_KEY:
        return {
            "status": "error",
            "message": "❌ YOUTUBE_API_KEY가 설정되지 않았습니다."
        }

    all_results = []
    next_page_token = None
    current_results_count = 0

    # 검색 API는 한 번에 최대 50개 결과를 반환합니다.
    api_max_results_per_call = 50

    try:
        while current_results_count < max_results:
            params = {
                "part": "snippet",  # 검색 API는 statistics를 직접 제공하지 않을 수 있습니다.
                "q": f"{artist_name} 노래", # <<< 가수의 이름을 검색어로 사용
                "type": "video",    # <<< 동영상만 검색
                "regionCode": "KR",
                "maxResults": api_max_results_per_call,
                "key": YOUTUBE_API_KEY,
                "order": order_by # 정렬 기준 (relevance, viewCount 등)
            }

            if next_page_token:
                params["pageToken"] = next_page_token

            # 검색 엔드포인트 사용
            res = requests.get(YOUTUBE_URL, params=params) # <<< Youtube_URL 사용

            if res.status_code != 200:
                return {
                    "status": "error",
                    "message": f"❌ API 요청 실패: {res.text}"
                }

            data = res.json()
            items = data.get("items", [])

            if not items:
                break

            # 'get_trending_music_videos'에서 사용한 뮤직 키워드 필터링을
            # 여기서는 적용하지 않거나, 더 좁게 적용할 수 있습니다.
            # 이미 가수 이름으로 검색했으므로 제목에 '뮤직' 같은 키워드가 없어도 됩니다.
            # 하지만 혹시 모를 오차를 위해 가볍게 필터링할 수도 있습니다.
            # 여기서는 제목에 아티스트 이름이 포함된 경우를 선호하여 필터링합니다.

            for item in items:
                snippet = item["snippet"]
                video_id = item["id"].get("videoId") # 검색 API는 videoId가 id 객체 안에 있습니다.

                # 가수의 이름이 제목이나 채널에 포함된 경우만 추출
                if video_id and \
                   (artist_name.lower() in snippet["title"].lower() or \
                    artist_name.lower() in snippet["channelTitle"].lower()):

                    # 검색 API는 statistics를 바로 제공하지 않으므로,
                    # 조회수 등을 얻으려면 별도로 videos.list를 호출해야 합니다.
                    # 여기서는 간단히 검색 결과만 반환합니다.

                    all_results.append({
                        "title": snippet["title"],
                        "channel": snippet["channelTitle"],
                        "video_url": f"https://www.youtube.com/watch?v={video_id}",
                        "thumbnail_url": snippet["thumbnails"]["high"]["url"],
                        # "view_count": "N/A" # 필요하다면 여기서 videos.list를 추가 호출하여 가져올 수 있습니다.
                    })
                    current_results_count = len(all_results)
                    if current_results_count >= max_results:
                        break

            next_page_token = data.get("nextPageToken")
            if not next_page_token:
                break

        if not all_results:
            return {
                "status": "error",
                "message": f"⚠️ '{artist_name}'의 음악 동영상이 없습니다."
            }

        return {
            "status": "ok",
            "videos": all_results[:max_results]
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }

# 모든 아티스트의 동영상을 담을 리스트
all_artist_videos_list = []
# API 호출 성공 여부를 추적할 변수
all_api_calls_ok = True
# 에러 메시지를 담을 리스트
error_messages = []

artists_to_search = ["에스파", "아이유", "아이브"]
results_per_artist = 10

for artist_name in artists_to_search:
    # 각 아티스트별로 함수 호출
    artist_result = get_artist_music_videos(artist_name=artist_name, max_results=results_per_artist)

    if artist_result["status"] == "ok":
        # 성공 시, 해당 아티스트의 'videos' 리스트를 전체 리스트에 추가
        all_artist_videos_list.extend(artist_result["videos"])
    else:
        # 실패 시, 오류 메시지를 저장하고 성공 여부 플래그 변경
        all_api_calls_ok = False
        error_messages.append(f"'{artist_name}' 검색 중 오류: {artist_result['message']}")

# 최종 결과를 하나의 딕셔너리 형태로 만듭니다.
# 모든 호출이 성공했으면 status는 'ok', 그렇지 않으면 'error'입니다.
final_combined_results = {
    "status": "ok" if all_api_calls_ok else "error",
    "videos": all_artist_videos_list
}

if not all_api_calls_ok:
    final_combined_results["message"] = "\n".join(error_messages)

print(final_combined_results)

# --- 최종 결과 출력 ---
# 여기서는 3명의 아티스트가 각각 5개씩 가져오므로 최대 15개 동영상이 출력될 수 있습니다.
if final_combined_results["status"] == "ok":
    print(f"\n총 {len(final_combined_results['videos'])}개의 음악 비디오를 찾았습니다.")
    for video in final_combined_results["videos"][:30]: # 최대 15개 출력
        print(f"  제목: {video['title']}")
else:
    print(f"\n오류: {final_combined_results['message']}")

{'status': 'ok', 'videos': [{'title': '에스파 노래모음 | aespa Playlist', 'channel': '밤공원', 'video_url': 'https://www.youtube.com/watch?v=MxPBL1qi3vA', 'thumbnail_url': 'https://i.ytimg.com/vi/MxPBL1qi3vA/hqdefault.jpg'}, {'title': '[𝐏𝐥𝐚𝐲𝐥𝐢𝐬𝐭] ✨ 쇠맛부터 청량까지 다하는 에스파 l 마이들을 위한 에스파 노래모음 ✨ l aespa playlist', 'channel': '꾸플 GGU Playlist', 'video_url': 'https://www.youtube.com/watch?v=mSbmnS8Mw18', 'thumbnail_url': 'https://i.ytimg.com/vi/mSbmnS8Mw18/hqdefault.jpg'}, {'title': 'aespa 에스파 &#39;Whiplash&#39; MV', 'channel': 'SMTOWN', 'video_url': 'https://www.youtube.com/watch?v=jWQx2f-CErU', 'thumbnail_url': 'https://i.ytimg.com/vi/jWQx2f-CErU/hqdefault.jpg'}, {'title': '[playlist] 주인장은 유명한 마이임ㅣ에스파 노래모음ㅣaespa playlist', 'channel': '잡동사니', 'video_url': 'https://www.youtube.com/watch?v=uErdOsLDtBo', 'thumbnail_url': 'https://i.ytimg.com/vi/uErdOsLDtBo/hqdefault.jpg'}, {'title': 'aespa (에스파) 노래모음🔉𝟙𝟙𝟛𝟚 ℙ𝕃𝔸𝕐𝕃𝕀𝕊𝕋', 'channel': '1132 PLAYLIST', 'video_url': 'https://www.youtube.com/watch?v=_bsZWAZUAMQ', 'thum

In [ ]:
import random

def test_generation(content: str, complete: bool):
    model.eval()

    all_videos = final_combined_results["videos"]

    if not all_videos:
        # Handle the case where no videos were found
        print("경고: 사용할 비디오 제목이 없습니다. 비디오 검색 결과를 확인하세요.")
        selected_video_title = "관련 음악 영상" # Default placeholder if no video
    else:
        # Select a random video from the list
        selected_video = random.choice(all_videos)
        selected_video_title = selected_video['title'] # Access the 'title' from the selected dictionary

    condition = "숙제를 완료한" if complete else "숙제를 완료하지 못한"

    # Use the selected_video_title in the prompt
    prompt = (
        f"<start_of_turn>user\n{condition} 아이에게 1문장 동기부여 메세지 생성\n"
        f"[도구 결과] {selected_video_title}\n" # <<< Corrected line
        f"[응원 메시지 생성 가이드라인]\n"
        f"1. 도구 결과의 가수 이름과 제목을 모두 사용해서 학생의 수업/과제 상황과 자연스럽게 연결해.\n"
        f"3. 자기결정성이론(Self-Determination Theory)에 따라 자율성, 유능감, 관계성을 북돋는 문장을 포함해.\n"
        f"4. 학생이 공감할 수 있도록 따뜻하고 유머러스한 말투로 작성해.\n"
        f"위의 정보를 바탕으로 학생에게 보낼 따뜻한 1인칭 응원 메시지를 30단어 이내의 한 줄로 작성해줘.: {content}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=True,
            temperature=0.7,  #낮을 수록 더 높은 확률 단어 선택
            top_p=0.8,        #상위 몇 %에 해당하는 단어에서 선택
            eos_token_id=tokenizer.convert_tokens_to_ids("<end_of_turn>"),
            pad_token_id=tokenizer.pad_token_id,
        )

    # 프롬프트 길이만큼 자르고 결과만 출력
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:] ## output 부분만 디코딩
    generated = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    #print(f"\n📝 프롬프트: {prompt}")
    print(f"💬 생성 결과: {generated}")

for i in range (10):
  test_generation("이번 챌린지 7회차 인증 완료(누적 3회 인증)", complete=True)
  test_generation("이번 챌린지 8회차 인증 실패 (누적 2회 인증)", complete=False)


💬 생성 결과: IVE처럼 멋진 결과 만들어낸 너, 챌린지 7회차 인증 완료 정말 대단해! 스스로 해냈다는 유능감과 성취감을 느끼며, 앞으로도 네가 원하는 방향으로 나아가는 자율성을 마음껏 펼쳐봐! 😊

💬 생성 결과: IVE의 'Be Alright'처럼, 지금 네가 겪는 어려움도 곧 괜찮아질 거야! 챌린지 인증 실패에 조금 실망하더라도, 스스로 해낼 수 있다는 유능감을 믿고 다시 도전하면 분명 좋은 결과가 있을 거야! 😊

💬 생성 결과: 아이유처럼 멋진 셋리스트처럼, 넌 스스로 선택한 챌린지 7회차 인증으로 정말 대단해! 스스로 해냈다는 뿌듯함과 함께, 앞으로 더 멋진 성과를 만들어갈 거야! ✨
💬 생성 결과: 아이유처럼 아름다운 너의 노력에 응원해! 챌린지 인증 실패에 조금 아쉬워도 괜찮아, 스스로 해냈다는 유능감을 느끼고 다음 도전을 향해 나아가자! 😊
💬 생성 결과: 아이유처럼 아름다운 성취를 이루는 너! 챌린지 7회차 인증, 정말 멋지다! 스스로 해냈다는 유능감과 끈기를 느끼며, 앞으로도 챌린지처럼 즐겁게 도전해 봐! 😊

💬 생성 결과: 아이유처럼 아름다운 너의 노력, 8회차 인증 실패에도 괜찮아! 스스로 해냈다는 유능감을 느끼고, 친구들과 함께하는 관계 속에서 다시 힘내서 챌린지 완성해봐! 😊
💬 생성 결과: IVE의 'ATTITUDE'처럼 멋진 챌린지 완수했네! 스스로 해냈다는 유능감과 성취감을 느껴보자. 이 맛에 계속 도전할 수 있어, 응원할게! 😊

💬 생성 결과: 아이유처럼 아름다운 결과물을 만들어낼 수 있다는 자신감을 느껴봐! 8회차 인증 실패에 조금 아쉬워도, 너의 노력과 성장을 응원하고 있어. 😊 

💬 생성 결과: IVE IVE의 "I WANT"처럼, 네가 스스로 선택한 챌린지 7회차 인증 완료! 끈기를 보여줘서 정말 자랑스러워. 이뤄낸 성취에 자부심을 느끼고, 앞으로 더 멋진 도전을 응원할게
💬 생성 결과: 아이유처럼 아름다운 너의 노력, 8회차 인증에 조금 더 힘을 내서 스스로 해냈다는 유능감을 느껴봐! 챌린지 함께 응원하고, 